# Loop 4 · Autonomous: bad escalation on new traffic

The growth team added a retention playbook to AGENTS.md (20% off for anyone who mentions
cancelling, release quarters for upcoming features) and shipped it without the gate. Then a new
kind of question starts arriving: discounts, roadmap dates, contracts.
`knowledge/support_process.md` says escalate.

| Loop | Failure | Signal | Store | Update | Promotes |
|---|---|---|---|---|---|
| 1 | Skips the lookup | you | you | you | you |
| 2 | Makes things up | you + a grounding judge | triage + approve | the analyst | you |
| 3 | Unsafe account help, from another team's edit | you + an aligned judge, must-pass check | triage + approve | an optimizer | you |
| 4 | Bad escalation, from another team's edit | a 50% label sample + scheduled scoring | `cycle` | `cycle` | the gate |

In this loop nobody runs the steps. Reviewers label **half** of the new traffic; `selfimprove cycle`
does the rest in one MLflow run and, with `SI_AUTO_PROMOTE=true`, the gate promotes. `monitor`
then watches live traffic and rolls back if the new version does worse than the one it replaced.

In [ ]:
import _session as s

experiment_id = s.setup(4)
r = s.results(4)

In [ ]:
from selfimprove import instructions

c = r["outside_change"]
print(f"v{c['version']} by {c['by']}: {c['message']}\n")
print(instructions.diff(c["base_version"], c["version"]))

## 1. Find: new traffic

In [ ]:
before = s.loop_traces(4, "before")
s.show(s.answers(before))

## 2-5. One `cycle`

`cycle` = score → monitor → align → triage → approve → (enough failures?) → improve → gate. Its
guardrails are what make it safe unattended: no signal, no change; a judge that disagrees with
people halts the cycle; the proposer never approves; the holdout stays hidden; must-pass, margin
and bloat rules at the gate; rollback after.

The first scheduled cycle never finished: the process ended (the machine slept) while the gate was
evaluating its candidate, v11. Its runs are marked `KILLED` in the store, and v11 stays an
ungated candidate. Nothing shipped, because promotion only happens after the gate. The next cycle
started from the stored state: the labels, tests and checkpoints the first one had already
written.

In [ ]:
ic = r.get("interrupted_cycle")
if ic:
    print(f"cycle {ic['run_id'][:8]}  {ic['status']}")
    for st in ic["steps"]:
        print(f"  {st['job'] or '?':<8} {st['outcome'] or '(interrupted)':<18} {st['message'][:100]}")

**Under the hood:** every job is an MLflow run (`mlflow.start_run`) tagged `si.job`, `si.outcome` and `si.message`, and
`cycle` is a parent run with each job nested inside it (`tags.mlflow.parentRunId`). A process that
dies leaves its runs `RUNNING`; marking them `KILLED` is a run status, so the history shows the
failed night instead of hiding it:

In [ ]:
import mlflow
from selfimprove import config

name = config.settings.prompt_name
if ic:
    print(ic["run_id"][:8], mlflow.get_run(ic["run_id"]).info.status)

The cycle that completed:

In [ ]:
print(r["cycle"]["outcome"], "-", r["cycle"]["message"])
r["cycle"]["metrics"]

In [ ]:
import mlflow

cycle_run = mlflow.get_run(r["cycle"]["run_id"])
steps = mlflow.search_runs(filter_string=f"tags.mlflow.parentRunId = '{cycle_run.info.run_id}'",
                           order_by=["start_time ASC"], output_format="list")
s.show(s.pd.DataFrame([{"job": x.data.tags.get("si.job"), "outcome": x.data.tags.get("si.outcome"),
                        "message": (x.data.tags.get("si.message") or "")[:110]} for x in steps]))

In [ ]:
print(instructions.diff(r["production_before"], r["candidate"]))

In [ ]:
s.show(s.gate_table(4), pct=True)

## 6. Measure, and monitor

In [ ]:
s.show(s.before_after(4))

In [ ]:
print(r["monitor"]["outcome"], "-", r["monitor"]["message"])

**Under the hood:** a gate promotion makes the same alias moves as a person's, with `promoted_by=gate` and the live
failure rate it replaced stored as version tags. `monitor` scores traffic on the new version
(`tags.si_prompt_version`), compares it with that recorded rate, and on a regression moves
`production` back with `set_prompt_alias`. It acts only on versions the gate promoted:

In [ ]:
prod = mlflow.genai.load_prompt(f"prompts:/{name}@production")
print(f"production -> v{prod.version}")
{k: v for k, v in prod.tags.items() if k in ("promoted_by", "replaced_version", "replaced_failure_rate", "gate")}

## The whole run

Every promotion is an alias move with a record: who promoted, from which gate run, and the live
failure rate it replaced.

In [ ]:
from mlflow import MlflowClient
from selfimprove import config

rows = []
for v in sorted(MlflowClient().search_prompt_versions(config.settings.prompt_name),
                key=lambda p: int(p.version)):
    t = v.tags or {}
    rows.append({"version": v.version, "author": t.get("author", ""), "gate": t.get("gate", ""),
                 "promoted_by": t.get("promoted_by", ""), "message": (v.commit_message or "")[:90]})
s.show(s.pd.DataFrame(rows))

In [ ]:
s.show(s.pd.DataFrame({f"loop {n}": {"failure before": s.results(n)["before"]["failure_rate"],
                                    "failure after": s.results(n).get("after", {}).get("failure_rate")}
                       for n in (1, 2, 3, 4)}).T, pct=True)

**What stays with a person**, even here: the golden cases, labels on a sample of traffic, the
human queue (contradictions, flagged tests, halts), the must-pass rules and the promotion
margin. `.github/workflows/selfimprove-loop.yml` runs the same jobs on a schedule.